In [ ]:
# Setup and helper functions moved to external module
from equations_file import *

# If you change the external module and want live reload during development, uncomment:
# %load_ext autoreload
# %autoreload 2

In [ ]:
with fitsio.FITS(galah_Gaia_fits) as hdul1:
    data_1 = hdul1[1].read()
galah_Gaia_raw = pd.DataFrame(data_1)
galah_Gaia_raw = ensure_native_endian(galah_Gaia_raw)

In [ ]:
galah_Gaia = galah_Gaia_raw[(galah_Gaia_raw['snr_px_ccd3'] > 30)       # Kushniruk 2026 also has cuts in log g and temp. I wonder if I should as well
                        & (galah_Gaia_raw['flag_sp'] == 0)
                        & (galah_Gaia_raw['flag_fe_h'] == 0)
                        & (galah_Gaia_raw['e_fe_h'] < 0.2)
                        & (galah_Gaia_raw['flag_sp_fit'] == 0)
                        & (galah_Gaia_raw['flag_red'] == 0)
                        & (galah_Gaia_raw['ruwe'] < 1.4)
                        & (galah_Gaia_raw['logg'] < 3.5)
                        & (galah_Gaia_raw['teff'] > 4000)
                        & (galah_Gaia_raw['teff'] < 6500)
                        # & (galah_Gaia_raw['flag_mg_fe'] == 0)
                        # & (galah_Gaia_raw['flag_na_fe'] == 0)
                        # & (galah_Gaia_raw['flag_cu_fe'] == 0)
                        & (galah_Gaia_raw['fe_h'] < -0.8)
                        # & (galah_Gaia_raw['fe_h'] < -1.25)
                        # & (galah_Gaia_raw['fe_h'] > -1.8)
                        # & (galah_Gaia_raw['ti_fe'] > 0.25)
                        
                        & (galah_Gaia_raw['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main',
                                                               'k2_hermes', 'galah_phase2', 'tess_hermes'
                                                               ]))
                        & (galah_Gaia_raw['radial_velocity'].notnull())
                        & (galah_Gaia_raw['ra_gaia'].notnull())
                        & (galah_Gaia_raw['dec_gaia'].notnull())
                        & (galah_Gaia_raw['pmra_gaia'].notnull())
                        & (galah_Gaia_raw['pmdec_gaia'].notnull())
                        & (galah_Gaia_raw['parallax_gaia'] > 0)
                        
    ]

print(f"Number of stars after cuts: {len(galah_Gaia)}")

In [ ]:
NGC3201_galah = pd.merge( galah_Gaia, Vmans_NGC3201_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC5139_galah = pd.merge( galah_Gaia, Vmans_NGC5139_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC1851_galah = pd.merge( galah_Gaia, Vmans_NGC1851_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')   
NGC0104_galah = pd.merge( galah_Gaia, Vmans_NGC0104_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')

galah_Kushniruk_GSE = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-35)**2) / (17.5**2)) + (((galah_Gaia['jphi'] + 130)**2) / (250**2)) < 1)
                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                 & (galah_Gaia_raw['flag_mg_fe'] == 0)
                                 & (galah_Gaia_raw['flag_na_fe'] == 0)
                                 & (galah_Gaia_raw['flag_cu_fe'] == 0)
                                 & ((galah_Gaia_raw['mg_fe'] - galah_Gaia_raw['cu_fe']) - ((1.06923) * galah_Gaia_raw['na_fe']) > 0.40150)
                                 ]

# Kushniruk et al . 2026 https://doi.org/10.1051/0004-6361/202451201
galah_Kushniruk_Thamnos1 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])- 13.5)**2) / (4.0**2)) + (((galah_Gaia['jphi'] + 600)**2) / (350**2)) < 1)
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                      ]

galah_Kushniruk_Thamnos2 = galah_Gaia[(((np.sqrt(galah_Gaia['jr'])-8.6)**2) / (3.5**2)) + (((galah_Gaia['jphi'] + 1184)**2) / (175**2)) < 1
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                      ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_GSE = galah_Gaia[GSE_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_Sequoia = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            # & (galah_Gaia_raw['y_fe'] < 0.8)
]

galah_Gaia_Halo = galah_Gaia[(np.sqrt((galah_Gaia['vphi']-230)**2 + (galah_Gaia['vr'])**2 + (galah_Gaia['vz'])**2) > 230) # Koppelman et al. 2019  https://doi.org/10.1051/0004-6361/201936738
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id'])) 
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id'])) 
                            # & (galah_Gaia_raw['fe_h'] < -1.1)
                            # & (galah_Gaia_raw['fe_h'] > -1.9)
                             ]


In [ ]:
galah_Gaia_else = galah_Gaia_raw[(galah_Gaia_raw['snr_px_ccd3'] > 30)       # Kushniruk 2026 also has cuts in log g and temp. I wonder if I should as well
                        & (galah_Gaia_raw['flag_sp'] == 0)
                        & (galah_Gaia_raw['flag_fe_h'] == 0)
                        & (galah_Gaia_raw['e_fe_h'] < 0.2)
                        & (galah_Gaia_raw['flag_sp_fit'] == 0)
                        & (galah_Gaia_raw['flag_red'] == 0)
                        & (galah_Gaia_raw['ruwe'] < 1.4)
                        & (galah_Gaia_raw['logg'] < 3.5)
                        & (galah_Gaia_raw['teff'] > 4000)
                        & (galah_Gaia_raw['teff'] < 6500)
                        # & (galah_Gaia_raw['fe_h'] < -0.75)
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id'])) 
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id'])) 
                        & (~galah_Gaia_raw['gaiadr3_source_id'].isin(galah_Gaia_Halo['gaiadr3_source_id']))
                        & (galah_Gaia_raw['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main',
                                                               'k2_hermes', 'galah_phase2', 'tess_hermes'
                                                               ]))
    ]

In [ ]:
Halo_Pradosh_galah_Gaia = galah_Gaia[
                                            # (galah_Gaia['fe_h'] < -0.8)
                                            (galah_Gaia['ti_fe'] > 0.25)
                                            & (galah_Gaia['flag_ti_fe'] == 0)
                                            & (galah_Gaia['Lz'] > -1500)
                                            & (galah_Gaia['Lz'] < 1500)
                                            & (galah_Gaia['ecc_gaia'] > 0.5)
                                            & (galah_Gaia['ecc_gaia'] < 0.8)
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]

thickdisk_galah_Gaia = galah_Gaia[(galah_Gaia['fe_h'] < -0.8)
                                                 & (galah_Gaia['ti_fe'] > 0.25)
                                                 & (galah_Gaia['Lz'] > 0)
                                                 & (galah_Gaia['ecc_gaia'] < 0.5)
                                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]
print(len(Halo_Pradosh_galah_Gaia))
print(len(thickdisk_galah_Gaia))

galah_Gaia_Halo = Halo_Pradosh_galah_Gaia

In [ ]:
"""Remember that the Pradosh Halo Is writen to override the other characterization for the Halo. I did this
because it was easier to do than to rewrite code for the Pradosh Halo, but I need to call it out if I want to ever use the other one"""

In [ ]:
print(f'len(NGC3201_galah): {len(NGC3201_galah)}')
print(f'len(NGC5139_galah): {len(NGC5139_galah)}')
print(f'len(NGC1851_galah): {len(NGC1851_galah)}')
print(f'len(NGC0104_galah): {len(NGC0104_galah)}')
print(f'len(galah_Kushniruk_GSE): {len(galah_Kushniruk_GSE)}')
print(f'len(galah_Kushniruk_Thamnos1): {len(galah_Kushniruk_Thamnos1)}')
print(f'len(galah_Kushniruk_Thamnos2): {len(galah_Kushniruk_Thamnos2)}')
print(f'len(galah_Gaia_GSE): {len(galah_Gaia_GSE)}')
print(f'len(galah_Gaia_Sequoia): {len(galah_Gaia_Sequoia)}')
print(f'len(galah_Gaia_Halo): {len(galah_Gaia_Halo)}')
print(f'len(galah_Gaia_else): {len(galah_Gaia_else)}')

In [ ]:
galah_Gaia_else_C = "#110C0C"
Sequoia_C = '#d62728'
GSE_C = "#16b823d8"
Halo_C = "#0e35e2"
thamnos_1_C = "#96bd0b"
thamnos_2_C = '#C43DFF'

In [ ]:
fig, ax = plt.subplots(2, 2, sharex=False, sharey=False, figsize=(22, 14.5)) # (rows, colums, share y axis and/or share x axis)
# fig.patch.set_alpha(0) # this can make the image transpearent.
fontsize = 30
labelsize = 20

ax[0,0].scatter(galah_Gaia_else['jphi']/1e3, galah_Gaia_else['energy']/1e5, label = 'All GALAH stars', s = 0.2, alpha = 0.3, c = galah_Gaia_else_C)
ax[0,0].scatter(galah_Gaia_Halo['jphi']/1e3, galah_Gaia_Halo['energy']/1e5, label = 'Milky Way Halo', s = 1, alpha = 0.5, c = Halo_C)
ax[0,0].scatter(galah_Kushniruk_Thamnos1['jphi']/1e3, galah_Kushniruk_Thamnos1['energy']/1e5, label = 'Thamnos 1', s = 10, alpha = 0.8, c = thamnos_1_C)
ax[0,0].scatter(galah_Kushniruk_Thamnos2['jphi']/1e3, galah_Kushniruk_Thamnos2['energy']/1e5, label = 'Thamnos 2', s = 10, alpha = 1.0, c = thamnos_2_C)
ax[0,0].scatter(galah_Gaia_GSE['jphi']/1e3, galah_Gaia_GSE['energy']/1e5, label = 'GSE', s = 7, alpha = 0.8, c = GSE_C)
ax[0,0].scatter(galah_Gaia_Sequoia['jphi']/1e3, galah_Gaia_Sequoia['energy']/1e5, label = 'Sequoia', s = 10, alpha = 0.8, c = Sequoia_C)
ax[0,0].set_xlim(-4000/1e3, 2500/1e3)
ax[0,0].set_ylim(-150000/1e5,30000/1e5)
ax[0,0].tick_params(axis='x', labelsize = labelsize)
ax[0,0].tick_params(axis='y', labelsize = labelsize)
ax[0,0].set_xlabel(r'$L_z$ ($10^3$ kpc km s$^{-1}$)', fontsize = fontsize)
ax[0,0].set_ylabel(r'Energy ($10^5$ km$^2$ s$^{-2}$)', fontsize = fontsize)
ax[0,0].text(0.02, 0.98, 'A', transform=ax[0,0].transAxes, fontsize=fontsize, fontweight='bold', va='top', ha='left')

handles, labels = ax[0,0].get_legend_handles_labels()

# Desired legend order
order = [
    labels.index('All GALAH stars'),
    labels.index('Milky Way Halo'),
    labels.index('GSE'),
    labels.index('Sequoia'),
    labels.index('Thamnos 1'),
    labels.index('Thamnos 2')
]

leg = ax[0,0].legend(
    [handles[i] for i in order],
    [labels[i] for i in order],
    fontsize=16,
    labelspacing=0.8,
    borderpad=1.2
)

for handle in leg.legend_handles:
    try:
        handle.set_sizes([100])
    except Exception:
        pass
###########################################################################################
ax[0,1].scatter(galah_Gaia_else['jphi']/1e3, np.sqrt(galah_Gaia_else['jr']), label = 'All Galah stars', s = 0.2, alpha = 0.3, c = galah_Gaia_else_C)
ax[0,1].scatter(galah_Gaia_Halo['jphi']/1e3, np.sqrt(galah_Gaia_Halo['jr']), label = 'Halo', s = 2, alpha = 1.0, c = Halo_C)
ax[0,1].scatter(galah_Kushniruk_Thamnos1['jphi']/1e3, np.sqrt(galah_Kushniruk_Thamnos1['jr']), label = 'Thamnos 1', s = 5, alpha = 0.5, c = thamnos_1_C)
ax[0,1].scatter(galah_Kushniruk_Thamnos2['jphi']/1e3, np.sqrt(galah_Kushniruk_Thamnos2['jr']), label = 'Thamnos 2', s = 10, alpha = 0.8, c = thamnos_2_C)
ax[0,1].scatter(galah_Gaia_GSE['jphi']/1e3, np.sqrt(galah_Gaia_GSE['jr']), label = 'GSE', s = 5, alpha = 0.8, c = GSE_C)
ax[0,1].scatter(galah_Gaia_Sequoia['jphi']/1e3, np.sqrt(galah_Gaia_Sequoia['jr']), label = 'Sequoia', s = 5, alpha = 0.8, c = Sequoia_C)
ax[0,1].set_xlim(-3000/1e3, 3000/1e3)
ax[0,1].set_ylim(0,60)
ax[0,1].tick_params(axis='x', labelsize = labelsize)
ax[0,1].tick_params(axis='y', labelsize = labelsize)
ax[0,1].set_xlabel(r'$L_z$ ($10^3$ kpc km s$^{-1}$)', fontsize = fontsize)
ax[0,1].set_ylabel(r'$\,\sqrt{J_r}\,$ (kpc km s$^{-1}$)$^{1/2}$', fontsize = fontsize);
ax[0,1].text(0.02, 0.98, 'B', transform=ax[0,1].transAxes, fontsize=fontsize, fontweight='bold', va='top', ha='left')

###########################################################################################
ax[1,0].scatter( galah_Gaia_else['jphi']/ galah_Gaia_else['jtot'], (galah_Gaia_else['jz'] - galah_Gaia_else['jr'])/galah_Gaia_else['jtot'], label = 'All Galah stars', s = 0.2, c = galah_Gaia_else_C, alpha = 0.3)
ax[1,0].scatter( galah_Gaia_Halo['jphi']/ galah_Gaia_Halo['jtot'], (galah_Gaia_Halo['jz'] - galah_Gaia_Halo['jr'])/galah_Gaia_Halo['jtot'], label = 'Milky Way Halo', s = 2, c = Halo_C, alpha = 0.7)
ax[1,0].scatter( galah_Kushniruk_Thamnos1['jphi']/ galah_Kushniruk_Thamnos1['jtot'], (galah_Kushniruk_Thamnos1['jz'] - galah_Kushniruk_Thamnos1['jr'])/galah_Kushniruk_Thamnos1['jtot'], label = 'Thamnos 1', c = thamnos_1_C, s = 5, alpha = 0.8)
ax[1,0].scatter( galah_Gaia_GSE['jphi']/ galah_Gaia_GSE['jtot'], (galah_Gaia_GSE['jz'] - galah_Gaia_GSE['jr'])/galah_Gaia_GSE['jtot'], label = 'GSE', c = GSE_C, s = 10, alpha  = 0.9)
ax[1,0].scatter( galah_Gaia_Sequoia['jphi']/ galah_Gaia_Sequoia['jtot'], (galah_Gaia_Sequoia['jz'] - galah_Gaia_Sequoia['jr'])/galah_Gaia_Sequoia['jtot'], label = 'Sequoia', c = Sequoia_C, s = 10, alpha = 0.9)
ax[1,0].scatter( galah_Kushniruk_Thamnos2['jphi']/ galah_Kushniruk_Thamnos2['jtot'], (galah_Kushniruk_Thamnos2['jz'] - galah_Kushniruk_Thamnos2['jr'])/galah_Kushniruk_Thamnos2['jtot'], label = 'Thamnos 2', c = thamnos_2_C, s = 10, alpha = 0.8)
ax[1,0].set_xlabel(r'$J_{phi}$ / $J_{tot}$', fontsize = fontsize)
ax[1,0].set_ylabel(r'( $J_z$ - $J_r$) / $J_{tot}$', fontsize = fontsize)
ax[1,0].text(0.02, 0.98, 'C', transform=ax[1,0].transAxes, fontsize=fontsize, fontweight='bold', va='top', ha='left')
ax[1,0].tick_params(axis='x', labelsize = labelsize)
ax[1,0].tick_params(axis='y', labelsize = labelsize)
############################################################################################
ax[1,1].scatter(galah_Gaia_else['vphi'], np.sqrt((galah_Gaia_else['vr'])**2 + (galah_Gaia_else['vz'])**2), label = 'All Galah stars', s = 0.2, alpha = 0.3, c = galah_Gaia_else_C)
ax[1,1].scatter(galah_Gaia_Halo['vphi'], np.sqrt((galah_Gaia_Halo['vr'])**2 + (galah_Gaia_Halo['vz'])**2), label = 'Halo', s = 2, alpha = 0.6, c = Halo_C)
ax[1,1].scatter(galah_Kushniruk_Thamnos1['vphi'], np.sqrt((galah_Kushniruk_Thamnos1['vr'])**2 + (galah_Kushniruk_Thamnos1['vz'])**2), label = 'Thamnos1', s = 5, alpha = 0.8, c = thamnos_1_C)
ax[1,1].scatter(galah_Kushniruk_Thamnos2['vphi'], np.sqrt((galah_Kushniruk_Thamnos2['vr'])**2 + (galah_Kushniruk_Thamnos2['vz'])**2), label = 'Thamnos2', s = 10, alpha = 0.9, c = thamnos_2_C)
ax[1,1].scatter(galah_Gaia_GSE['vphi'], np.sqrt((galah_Gaia_GSE['vr'])**2 + (galah_Gaia_GSE['vz'])**2), label = 'GSE', s = 10, alpha = 0.9, c = GSE_C)
ax[1,1].scatter(galah_Gaia_Sequoia['vphi'], np.sqrt((galah_Gaia_Sequoia['vr'])**2 + (galah_Gaia_Sequoia['vz'])**2), label = 'Sequoia', s = 10, alpha = 0.9, c = Sequoia_C)
ax[1,1].set_xlim(-400, 400)
ax[1,1].set_ylim(0,400)
ax[1,1].tick_params(axis='x', labelsize = labelsize)
ax[1,1].tick_params(axis='y', labelsize = labelsize)
ax[1,1].set_xlabel(r'$V_{phi}$ (km s$^{-1}$)', fontsize = fontsize)
ax[1,1].set_ylabel(r'$\sqrt{V_r^2 + V_z^2}$ (km s$^{-1}$)', fontsize = fontsize)
ax[1,1].text(0.02, 0.98, 'D', transform=ax[1,1].transAxes, fontsize=fontsize, fontweight='bold', va='top', ha='left')

# ax[1,1].scatter(galah_Gaia_else['jphi'], galah_Gaia_else['ecc_gaia'], label = 'All Galah stars', s = 0.2, alpha = 0.3, c = galah_Gaia_else_C)
# ax[1,1].scatter(galah_Gaia_Halo['jphi'], galah_Gaia_Halo['ecc_gaia'], label = 'Halo', s = 2, alpha = 0.6, c = Halo_C)

# plt.savefig('Kinematics.png', dpi = 1200, bbox_inches='tight')




In [ ]:
plt.figure(figsize=(11, 6)) #(length, height)
elm_ratio = 'fe_h'
n_components = 1
shift_right = 0.32
fontsize = 20

elm_lable = elm_ratio
elm_lable_display = '/'.join([seg.capitalize() for seg in elm_lable.replace('_', '/').split('/')])

sec_mean = np.mean(galah_Gaia_Sequoia[elm_ratio])
one_sig_sec = np.std(galah_Gaia_Sequoia[elm_ratio])
pl_1sig_sec = sec_mean + one_sig_sec
min_1sig_sec = sec_mean - one_sig_sec
plot_gmm(galah_Gaia_Sequoia[elm_ratio].dropna(), f'Sequoia', n_components = n_components, color=Sequoia_C, maximum_peak=True, plot_compents = True, fontsize = fontsize, 
        #  shift_right=shift_right, shift_up= 0.91)
        shift_right=-0.25, shift_up= 0.2, linewidth_main = 5)
plt.axvspan(min_1sig_sec, pl_1sig_sec, facecolor=Sequoia_C, alpha=0.2, label = r"$\pm 1\sigma_{\mathrm{Sec}}$")

galah_Gaia_Halo_sec_1sig = ((galah_Gaia_Halo[elm_ratio] < pl_1sig_sec) & (galah_Gaia_Halo[elm_ratio] > min_1sig_sec))
plt.text(sec_mean -0.28, 1.28, f'{len(galah_Gaia_Halo[galah_Gaia_Halo_sec_1sig])} Halo stars in 'r"$\pm 1 \sigma_{\mathrm{Sec}}$", c = Halo_C, fontsize = 13.0)



gse_mean = np.mean(galah_Gaia_GSE[elm_ratio])
one_sig_gse = np.std(galah_Gaia_GSE[elm_ratio])
pl_1sig_gse = gse_mean + one_sig_gse
min_1sig_gse = gse_mean - one_sig_gse
plot_gmm(galah_Gaia_GSE[elm_ratio].dropna(), f'GSE', n_components = n_components, color=GSE_C, maximum_peak=True, fontsize = fontsize, 
        #  shift_right=shift_right, shift_up= 0.81)
        shift_right=-0.23, shift_up= 0.4, linewidth_main = 5)
plt.axvspan(min_1sig_gse, pl_1sig_gse, facecolor=GSE_C, alpha=0.2, label = r"$\pm 1\sigma_{\mathrm{GSE}}$", zorder=0,)

galah_Gaia_Halo_gse_1sig = ((galah_Gaia_Halo[elm_ratio] < pl_1sig_gse) & (galah_Gaia_Halo[elm_ratio] > min_1sig_gse))
plt.text(gse_mean -0.28, 1.55, f'{len(galah_Gaia_Halo[galah_Gaia_Halo_gse_1sig])} Halo stars in 'r"$\pm 1\sigma_{\mathrm{GSE}}$", c = Halo_C, fontsize = 13.0)


# plot_gmm(galah_Gaia_Halo[elm_ratio].dropna(), f'Halo', n_components = n_components, color='blue', maximum_peak=True, fontsize = fontsize, shift_right=shift_right, shift_up= 0.71)

# plt.hist(galah_Gaia_Sequoia[elm_ratio].dropna(), bins=30, color = 'r', alpha=0.5, label='Sequoia', density=True, histtype='step')
# plt.hist(galah_Gaia_GSE[elm_ratio].dropna(), bins=30, color = 'g', density=True, histtype='step')
# plt.hist(galah_Gaia_Halo[elm_ratio].dropna(), bins=30, color = Halo_C, alpha=0.5, label='Halo', density=True, histtype='step', linewidth = 5)

# galah_Kushniruk_Thamnos1_mask = (galah_Kushniruk_Thamnos1['fe_h'] <-0.75)
# plt.hist(galah_Kushniruk_Thamnos1[galah_Kushniruk_Thamnos1_mask][elm_ratio].dropna(), bins=30, color = 'darkseagreen', alpha=0.5, label='Thamnos', density=True, histtype='step')
# plot_gmm(galah_Kushniruk_Thamnos1[galah_Kushniruk_Thamnos1_mask][elm_ratio].dropna(), f'Halo', n_components = 3, color='darkseagreen', maximum_peak=True, fontsize = fontsize, shift_right=shift_right, shift_up= 0.61)

leg = plt.legend(fontsize=18, labelspacing=0.8, borderpad=1.2, loc = 'upper left')
# for handle in leg.legend_handles:
    # handle.set_sizes([100])

peak_height  = max(plt.gca().get_ylim())
plt.ylim(0, peak_height + 0.5)
plt.tick_params(axis='x', labelsize=20)
plt.tick_params(axis='y', labelsize=20)
plt.xlabel(f'[{elm_lable_display}]', fontsize = 30)
plt.ylabel('Density', fontsize = 30)


# plt.savefig('Sec and GSE metalicity.png', dpi = 2400, bbox_inches='tight')


In [ ]:

element_list_partial_small_galah_7 = [ 'fe', 'cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_2 = ['cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_3 = ['fe', 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_4 = ['fe', 'si', 'nd', 'na', 'mg', 'k', 'ca', 'sc', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba']
element_list_partial_small_galah_7_5 = [ 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']  # now I wonder if I could use sm_fe, o_fe, 
element_list_partial_small_galah_10 = ['fe', 'v', 'sc', 'si', 'ca', 'mn', 'na', 'nd']                                                             # and/or la_fe
element_list_partial_small_galah_11 = ['y', 'ba', 'nd', 'eu']                                                                                 # also la and sm would bring me to (50, 45, or 44combined)/65 sequoia stars otherwise
element_list_partial_small_galah_16 =  ["na", "mg", "ca", "ti", "cr" ,"mn", "ni", 'cu', "y", "ba", "nd"]


ASA_Plots_ratios = ['fe', 'si', 'ca', 'ti', 'mn', 'ni', 'y', 'nd']
S_VS_G_best_elms = ['na', 'mn', 'ti', 'cr']

# element_list = S_VS_G_best_elms
element_list = element_list_partial_small_galah_7_5


galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia, element_list)

galah_Gaia_new_ratios = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0] # this ontains a full catalog from galah but now it includes ratios for Ca, Si, Ti, Ni, Zr, Ce, and Nd relative to eachother: Pradosh did recomend taking out
                                                                 # Si, and Ni whitch I can do by changing "element_list_partial_large" to element_list_partial_small
galah_gaia_new_element_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]     # this just lists the new columns ratios so I can more easily call them
galah_gaia_new_element_ratio_errors_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[2] 

galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_GSE, element_list)
galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_GSE, element_list)
galah_Gaia_Kushniruk_GSE_new_ratios = galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia, element_list)
galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC3201_galah, element_list)
# galah_Gaia_NGC3201_new_ratios = galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC5139_galah, element_list)
galah_Gaia_NGC5139_new_ratios = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos1, element_list)
galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos2, element_list)
galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC1851_galah, element_list)
# galah_Gaia_NGC1851_new_ratios = galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Halo, element_list)
galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(Halo_Pradosh_galah_Gaia, element_list)
Halo_Pradosh_galah_Gaia_new_ratios = Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_else_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_else, element_list)
galah_Gaia_else_new_ratios = galah_Gaia_else_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_raw_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_raw, element_list)
galah_Gaia_raw_new_ratios = galah_Gaia_raw_new_ratios_catolog_ratiolist_ratioerrorlist[0]


'----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'

# element_list_full_galah_almost = ["n", "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", "y", "zr", "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.
element_list_full_galah_almost = ['fe', 'c', 'n', "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", 
                                  'rb', 'sr', 
                                  "y", "zr", 
                                  'mo', 'ru',
                                  "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.


element_list_nessessary_flags = element_list_full_galah_almost

galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia, element_list_nessessary_flags)
galah_Gaia_new_ratios_nessessary_flags_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]
galah_Gaia_new_ratios_nessessary_flags = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Sequoia, element_list_nessessary_flags)
galah_Gaia_Sequoia_new_ratios_nessessary_flags = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]
 
galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_GSE, element_list_nessessary_flags)
galah_Gaia_GSE_new_ratios_nessessary_flags = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos_1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Kushniruk_Thamnos1, element_list_nessessary_flags)
galah_Gaia_Thamnos_1_new_ratios_nessessary_flags = galah_Gaia_Thamnos_1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos_2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Kushniruk_Thamnos2, element_list_nessessary_flags)
galah_Gaia_Thamnos_2_new_ratios_nessessary_flags = galah_Gaia_Thamnos_2_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Halo, element_list_nessessary_flags)
galah_Gaia_Halo_new_ratios_nessessary_flags = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(Halo_Pradosh_galah_Gaia, element_list_nessessary_flags)
Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags = Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]



In [ ]:
num_bins = 50
percentiles = [16, 50, 84]

# elm_ratios = ['nd_rb', 'sm_rb', 'eu_rb', 'nd_y', 'sm_y', 'eu_y','nd_zr', 'sm_zr', 'eu_zr',] # r/s1 # all show GSE having a higher dex
# elm_ratios = ['nd_ba', 'sm_ba', 'eu_ba', 'nd_la', 'sm_la', 'eu_la','nd_ce', 'sm_ce', 'eu_ce',] # r/s2 Ba acts like s1, la and ce act like r
elm_ratios = ['ba_rb', 'la_rb', 'ce_rb', 'ba_y', 'la_y', 'ce_y', 'ba_zr', 'la_zr', 'ce_zr',] # s2/s1 # again Ba acts like s1 while la and ce act like r

elm_ratios = ['fe_h']

# Datasets mapping: name -> (dataframe_with_columns, color)
# datasets = {
#     'Sec': (galah_Gaia_Sequoia_new_ratios, Sequoia_C),
#     'GSE': (galah_Gaia_GSE_new_ratios, GSE_C),
#     'Halo': (Halo_Pradosh_galah_Gaia_new_ratios, Halo_C),
#     # 'Tham1': (galah_Gaia_Thamnos_1_new_ratios, thamnos_1_C),
#     # 'Tham2': (galah_Gaia_Thamnos_2_new_ratios, thamnos_2_C),
# }

datasets = {
    'Halo': (Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags, Halo_C),
    'GSE': (galah_Gaia_GSE_new_ratios_nessessary_flags, GSE_C),
    # 'Seq': (galah_Gaia_Sequoia_new_ratios_nessessary_flags, Sequoia_C),
    # 'Tham1': (galah_Gaia_Thamnos_1_new_ratios_nessessary_flags, thamnos_1_C),
    # 'Tham2': (galah_Gaia_Thamnos_2_new_ratios_nessessary_flags, thamnos_2_C),
}

# Reference dataset to set the edges/range (use 'Halo' as default)
ref = 'Halo'
for elm_ratio in elm_ratios:
    data, results, edges, min_value, max_value = kde_and_percentiles(datasets, ref, elm_ratio, num_bins = 50, percentiles = [16, 50, 84])


    fig, ax = plt.subplots(2, 1, sharex=False, sharey=False, figsize=(11, 12)) # (rows, colums, share y axis and/or share x axis)
    fontsize = 30
    labelsize = 25

    elm_ratio = elm_ratio
    elm_lable = elm_ratio
    elm_lable_display = '/'.join([seg.capitalize() for seg in elm_lable.replace('_', '/').split('/')])

    n_components = 1
    shift_right = 0.32
    'Gausian Curve for Sequoia'


    # sec_mean = np.mean(galah_Gaia_Sequoia[elm_ratio])
    # one_sig_sec = np.std(galah_Gaia_Sequoia[elm_ratio])
    # pl_1sig_sec = sec_mean + one_sig_sec
    # min_1sig_sec = sec_mean - one_sig_sec
    # plot_gmm(galah_Gaia_Sequoia[elm_ratio].dropna(), f'Sequoia', n_components = n_components, color=Sequoia_C, maximum_peak=True, plot_compents = True, fontsize = fontsize, 
    #         #  shift_right=shift_right, shift_up= 0.91)
    #         shift_right=-0.25, shift_up= 0.2, linewidth_main = 5,
    #         ax = ax[0])
    # ax[0].axvspan(min_1sig_sec, pl_1sig_sec, facecolor=Sequoia_C, alpha=0.2, label = r"$\pm 1\sigma_{\mathrm{Sec}}$")

    # galah_Gaia_Halo_sec_1sig = ((galah_Gaia_Halo[elm_ratio] < pl_1sig_sec) & (galah_Gaia_Halo[elm_ratio] > min_1sig_sec))
    # ax[0].text(sec_mean -0.28, 1.28, f'{len(galah_Gaia_Halo[galah_Gaia_Halo_sec_1sig])} Halo stars in 'r"$\pm 1 \sigma_{\mathrm{Sec}}$", c = Halo_C, fontsize = 13.0)


    'Gausian Curve for GSE'
    # gse_mean = np.mean(galah_Gaia_GSE[elm_ratio])
    # one_sig_gse = np.std(galah_Gaia_GSE[elm_ratio])
    # pl_1sig_gse = gse_mean + one_sig_gse
    # min_1sig_gse = gse_mean - one_sig_gse
    # plot_gmm(galah_Gaia_GSE[elm_ratio].dropna(), f'GSE', n_components = n_components, color=GSE_C, maximum_peak=True, fontsize = fontsize, 
    #         #  shift_right=shift_right, shift_up= 0.81)
    #         shift_right=-0.23, shift_up= 0.4, linewidth_main = 5,
    #         ax = ax[0])
    # ax[0].axvspan(min_1sig_gse, pl_1sig_gse, facecolor=GSE_C, alpha=0.2, label = r"$\pm 1\sigma_{\mathrm{GSE}}$", zorder=0,)

    # galah_Gaia_Halo_gse_1sig = ((galah_Gaia_Halo[elm_ratio] < pl_1sig_gse) & (galah_Gaia_Halo[elm_ratio] > min_1sig_gse))
    # ax[0].text(gse_mean -0.28, 1.55, f'{len(galah_Gaia_Halo[galah_Gaia_Halo_gse_1sig])} Halo stars in 'r"$\pm 1\sigma_{\mathrm{GSE}}$", c = Halo_C, fontsize = 13.0)


    'Gausian Curve for Halo'
    # plot_gmm(galah_Gaia_Halo[elm_ratio].dropna(), f'Halo', n_components = n_components, color='blue', maximum_peak=True, fontsize = fontsize, shift_right=shift_right, shift_up= 0.71)

    ####################################################################
    'Histograms'

    for name, (d, c, label, percentiles_label) in data.items():
        # compute shared bin edges from all samples (uses 30 bins)
        all_vals = np.concatenate([t[0].dropna().values for t in data.values()])
        shared_bins = np.linspace(np.nanmin(all_vals), np.nanmax(all_vals), 31)

        counts, bins = np.histogram(d.dropna().values, bins=shared_bins)
        # Normalise so the highest bin = 1
        counts = counts / counts.max() if counts.size and counts.max() > 0 else counts

        ax[0].stairs(
            counts,
            bins,
            color=c,
            # label=label,
            linewidth=2,
            alpha = 0.5
        )

    #####################################################################
    'Kernal Density estimates'

    for name, (d, c, label, percentiles_label) in data.items():
        x = d.dropna().values

        # sklearn expects shape (n_samples, n_features)
        x_2d = x.reshape(-1, 1)

        bandwidth_percent = 0.05
        bandwidth_range = np.ptp(x)  # full observed range of the sample
        bandwidth = max(bandwidth_range * bandwidth_percent, 1e-6)
        print(bandwidth)
        kde = KernelDensity(
            kernel='gaussian',
            bandwidth=bandwidth
        )
        
        kde.fit(x_2d)

        x_grid = np.linspace(x.min(), x.max(), 1000).reshape(-1, 1)

        density = np.exp(kde.score_samples(x_grid))
        # normalize so the peak density for this sample is 1
        max_density = density.max() if density.size else 0.0
        if max_density > 0:
            density = density / max_density

        ax[0].plot(
            x_grid[:, 0],
            density,
            color=c,
            linewidth=5,
            label= label
        )
        
    ######################################################################
    'labels'
    ax[0].text(0.02, 0.98, 'A', transform=ax[0].transAxes, fontsize=fontsize+5, fontweight='bold', va='top', ha='left')
    leg = ax[0].legend(fontsize=fontsize - 5, labelspacing=0.8, borderpad=1.2, frameon=False, loc='best'
                    #, bbox_to_anchor=(0, 0.6)
                    )
    # for handle in leg.legend_handles:
        # handle.set_sizes([100])

    peak_height  = max(ax[0].get_ylim())
    ax[0].set_ylim(0, peak_height * 1.1)
    ax[0].tick_params(axis='x', labelsize=labelsize)
    ax[0].tick_params(axis='y', labelsize=labelsize)
    ax[0].set_xlim(min_value,max_value)


    ax[0].set_ylabel('Density', fontsize = fontsize)

    #########################################################################################

    linewidth = 5
    ax[1].axhline(50, linestyle='solid', linewidth=3, alpha=0.4, c='grey')
    # ax[1].axhline(16, linestyle='solid', linewidth=3, alpha=0.4, c='grey')
    # ax[1].axhline(84, linestyle='solid', linewidth=3, alpha=0.4, c='grey')

    # Plot cumulative percent curves from results
    for name, info in results.items():
        perc = info.get('percents')
        total = info.get('total', 0)
        color = info.get('color', '#000000')
        if perc is not None and edges is not None:
            ax[1].plot(edges, perc, color=color, linewidth=linewidth, label=f"{name} N:{total}", alpha=0.85)

    # Vertical lines for percentiles per dataset
    linestyles = {50: ':', 
                #   16: '-.', 
                #   84: ':'
                }
    for name, info in results.items():
        pts = info.get('points', {})
        color = info.get('color', '#000000')
        for p, ls in linestyles.items():
            val = pts.get(p, np.nan)
            if not np.isnan(val):
                ax[1].axvline(val, color=color, linestyle=ls, linewidth=3, alpha=0.6)

    # Labels, ticks, legend
    elm_lable = elm_ratio
    elm_lable_display = '/'.join([seg.capitalize() for seg in elm_lable.replace('_', '/').split('/')])
    ax[1].set_xlabel(f'[{elm_lable_display}]', fontsize = fontsize)
    ax[1].set_ylabel('Detection %', fontsize = fontsize)
    ax[1].tick_params(axis='x', labelsize=labelsize)
    ax[1].tick_params(axis='y', labelsize=labelsize)
    ax[1].text(0.02, 0.98, 'B', transform=ax[1].transAxes, fontsize=fontsize+5, fontweight='bold', va='top', ha='left')
    leg = ax[1].legend(fontsize=fontsize-5, labelspacing=0.6, borderpad=0.8, frameon=False, loc='best')
    ax[1].set_ylim(0,105)
    ax[1].set_xlim(min_value, max_value)

    ax[0].axvline(-1.2, color=color, linestyle=ls, linewidth=3, alpha=0.6)
    plt.tight_layout()


In [ ]:
num_bins = 50
percentiles = [16, 50, 84]

elm_ratios = ['nd_rb', 'sm_rb', 'eu_rb', 'nd_y', 'sm_y', 'eu_y','nd_zr', 'sm_zr', 'eu_zr',] # r/s1 # all show GSE having a higher dex
# elm_ratios = ['nd_ba', 'sm_ba', 'eu_ba', 'nd_la', 'sm_la', 'eu_la','nd_ce', 'sm_ce', 'eu_ce',] # r/s2 Ba acts like s1, la and ce act like r
# elm_ratios = ['ba_rb', 'la_rb', 'ce_rb', 'ba_y', 'la_y', 'ce_y', 'ba_zr', 'la_zr', 'ce_zr',] # s2/s1 # again Ba acts like s1 while la and ce act like r

# elm_ratios = ['fe_h']

# Datasets mapping: name -> (dataframe_with_columns, color)
# datasets = {
#     'Sec': (galah_Gaia_Sequoia_new_ratios, Sequoia_C),
#     'GSE': (galah_Gaia_GSE_new_ratios, GSE_C),
#     'Halo': (Halo_Pradosh_galah_Gaia_new_ratios, Halo_C),
#     # 'Tham1': (galah_Gaia_Thamnos_1_new_ratios, thamnos_1_C),
#     # 'Tham2': (galah_Gaia_Thamnos_2_new_ratios, thamnos_2_C),
# }

datasets = {
    'Halo': (Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags, Halo_C),
    'GSE': (galah_Gaia_GSE_new_ratios_nessessary_flags, GSE_C),
    # 'Seq': (galah_Gaia_Sequoia_new_ratios_nessessary_flags, Sequoia_C),
    # 'Tham1': (galah_Gaia_Thamnos_1_new_ratios_nessessary_flags, thamnos_1_C),
    # 'Tham2': (galah_Gaia_Thamnos_2_new_ratios_nessessary_flags, thamnos_2_C),
}

# Reference dataset to set the edges/range (use 'Halo' as default)
ref = 'Halo'
fig, ax = plt.subplots(3, 3, sharex=False, sharey=False, figsize=(17, 14)) # (rows, colums, share y axis and/or share x axis)
fig.subplots_adjust(hspace=0.35, wspace=0.45)
# plt.tight_layout()


for i in range(len(elm_ratios)):
    elm_ratio = elm_ratios[i]
    j = i%3
    k = np.floor(i / 3).astype(int)
    print(f'{i} and {j} and {k}')
    data, results, edges, min_value, max_value = kde_and_percentiles(datasets, ref, elm_ratio, num_bins = 50, percentiles = [16, 50, 84])


    fontsize = 30
    labelsize = 25

    elm_ratio = elm_ratio
    elm_lable = elm_ratio
    elm_lable_display = '/'.join([seg.capitalize() for seg in elm_lable.replace('_', '/').split('/')])

    n_components = 1
    shift_right = 0.32
    'Gausian Curve for Sequoia'


    ####################################################################
    'Histograms'

    for name, (d, c, label, percentiles_label) in data.items():
        # compute shared bin edges from all samples (uses 30 bins)
        all_vals = np.concatenate([t[0].dropna().values for t in data.values()])
        shared_bins = np.linspace(np.nanmin(all_vals), np.nanmax(all_vals), 31)

        counts, bins = np.histogram(d.dropna().values, bins=shared_bins)
        # Normalise so the highest bin = 1
        counts = counts / counts.max() if counts.size and counts.max() > 0 else counts

        ax[j,k].stairs(
            counts,
            bins,
            color=c,
            # label=label,
            linewidth=2,
            alpha = 0.5
        )

    #####################################################################
    'Kernal Density estimates'

    for name, (d, c, label, percentiles_label) in data.items():
        x = d.dropna().values

        # sklearn expects shape (n_samples, n_features)
        x_2d = x.reshape(-1, 1)

        bandwidth_percent = 0.08
        bandwidth_range = np.ptp(x)  # full observed range of the sample
        bandwidth = max(bandwidth_range * bandwidth_percent, 1e-6)
        print(bandwidth)
        kde = KernelDensity(
            kernel='gaussian',
            bandwidth=bandwidth
        )
        
        kde.fit(x_2d)

        x_grid = np.linspace(x.min(), x.max(), 1000).reshape(-1, 1)

        density = np.exp(kde.score_samples(x_grid))
        # normalize so the peak density for this sample is 1
        max_density = density.max() if density.size else 0.0
        if max_density > 0:
            density = density / max_density

        ax[j,k].plot(
            x_grid[:, 0],
            density,
            color=c,
            linewidth=5,
            # label= label
            label = percentiles_label
        )
        
    ######################################################################
    'labels'
    leg = ax[j,k].legend(fontsize=9, labelspacing=0.8, borderpad=1.2, frameon=False, 
                         loc='upper left'
                    , bbox_to_anchor=(-0.04, 1.055)
                    )
    # for handle in leg.legend_handles:
        # handle.set_sizes([100])

    peak_height  = max(ax[j,k].get_ylim())
    ax[j,k].set_ylim(0, peak_height * 1.1)
    ax[j,k].tick_params(axis='x', labelsize=labelsize)
    ax[j,k].tick_params(axis='y', labelsize=labelsize)
    ax[j,k].set_xlim(min_value,max_value)


    ax[j,k].set_ylabel('Density', fontsize = fontsize-10)
    ax[j,k].set_xlabel(f'{elm_ratio}', fontsize = fontsize-10)


